# RQ2 — fine-tuning curve on NISAR chips (issue #24)

How many labelled NISAR chips does it take to recover what the Sentinel-1 model
loses on L-band? This notebook fine-tunes a Sen1Floods11 source model on
n = 0, 5, 10, 20, 35 NISAR chips across 3 seeds and plots the curve with error bars.

**Runtime:** roughly 25 minutes for the ResNet-50 arm on a T4; the MaxViT arm is
about four times that. The whole sweep fits in one Colab session.

### Two design choices worth knowing before you read the numbers

**A fixed step budget, not epochs.** Every point trains for the same 300 optimiser
steps. Counting epochs instead would give n=35 seven times the updates of n=5, and
the curve would confound *more data* with *more training*.

**No checkpoint selection.** The 35-chip pool and the 20-chip test set are the only
labelled NISAR data that exists, so there is no third split to pick an epoch on.
Selecting the best epoch by test score would report a number tuned on the chips it
claims to generalise to. Each run is scored as it stands when the budget runs out.

The split is fixed in `data/labels/rq2_splits.json` and explained in
`docs/rq2_splits.md`. Do not re-roll it: the RQ1 zero-shot run (#22) is scored on
the same 20 chips, and that is what makes its number the n=0 point of this curve.

## 1. Code

In [ ]:
import os, shutil

REPO   = "https://github.com/cs59300-deep-learning/nisar-flood-mapping.git"
BRANCH = "feat/24-rq2-splits"   # "main" once the PR is merged

# Clone outside any previous clone, so re-running this cell replaces it
# instead of nesting a second copy inside the first.
BASE = next(d for d in ["/content", "/kaggle/working", os.getcwd()] if os.path.isdir(d))
os.chdir(BASE)
shutil.rmtree("nisar-flood-mapping", ignore_errors=True)
!git clone --quiet --branch $BRANCH --depth 1 $REPO
os.chdir("nisar-flood-mapping")
!git log --oneline -1

The scaffold commit created every source file empty, and Python imports an empty
module without complaint. This cell fails loudly on the wrong branch rather than
silently producing no results five cells later.

In [ ]:
import pathlib

for f in ["src/nisar_flood/experiments/finetune_curve.py",
          "src/nisar_flood/datasets/nisar_chips.py",
          "scripts/make_rq2_splits.py",
          "configs/finetune.yaml",
          "data/labels/rq2_splits.json"]:
    size = pathlib.Path(f).stat().st_size
    assert size > 0, f"{f} is empty - the clone is on a branch without the implementation"
    print(f"{size:>8,} bytes  {f}")

## 2. Hardware

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"{name} | {vram:.0f} GB | bfloat16: {torch.cuda.is_bf16_supported()}")
    if torch.cuda.get_device_capability(0)[0] >= 8:
        torch.set_float32_matmul_precision("high")   # TF32 matmuls on A100/L4/H100
        print("TF32 enabled")
else:
    print("No GPU. Runtime > Change runtime type > T4 GPU, then rerun from cell 1.")

## 3. Dependencies

Colab has torch already. `segmentation_models_pytorch` brings the U-Net decoder and
the timm encoders; `rasterio` reads the chips.

In [ ]:
!pip install --quiet segmentation-models-pytorch rasterio huggingface_hub
print("installed")

## 4. Sign in to Hugging Face

The chips, the label masks and the source checkpoints all live in the private
`nisar-flood-mapping` organisation, so this step is required.

The widget below asks for **your own** access token (hf.co/settings/tokens, read
scope is enough). Type it into the widget — do not paste it into a cell, or it gets
saved into the notebook file and travels with every copy.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 5. Pick the arm

Both arms run through the same code; only the source config and checkpoint differ.
Run the whole notebook once per arm.

The MaxViT checkpoint is Nidhi's #20. Check with her that it is final before
reporting its curve — at the time of writing its repo has the weights but no
`eval.json` or `summary.json`, which the ResNet-50 repo does have.

In [ ]:
ARM = "resnet50"           # "resnet50" or "maxvit"

ARMS = {
    "resnet50": {"hf": "nisar-flood-mapping/unet-resnet50-sen1floods11",
                 "config": "configs/unet_resnet50.yaml",
                 "run": "runs/unet_resnet50/seed0/best.pt"},
    "maxvit":   {"hf": "nisar-flood-mapping/unet-maxvit-sen1floods11",
                 "config": "configs/unet_maxvit.yaml",
                 "run": "runs/unet_maxvit/seed0/best.pt"},
}
arm = ARMS[ARM]
OUT = f"runs/rq2_finetune_{ARM}"
print(ARM, "->", arm["hf"])

## 6. Data

55 chips and their label masks, about 115 MB. The chips are two-band GeoTIFFs
holding HH and HV gamma0 in dB; the masks are 1 water, 0 dry, 2 suspected flooded
vegetation, -1 unusable.

In [ ]:
from huggingface_hub import snapshot_download
import shutil, pathlib

local = snapshot_download(repo_id="nisar-flood-mapping/nisar-flood-masks-amazon",
                          repo_type="dataset",
                          allow_patterns=["chips/*", "labels/masks/*"])

for src, dst in [("chips", "data/chips"), ("labels/masks", "data/labels/masks")]:
    pathlib.Path(dst).parent.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(pathlib.Path(local) / src, dst)

chips = sorted(pathlib.Path("data/chips").glob("*.tif"))
masks = sorted(pathlib.Path("data/labels/masks").glob("*_label.tif"))
print(f"{len(chips)} chips, {len(masks)} masks")
assert len(chips) == len(masks) == 55, "expected 55 of each"

## 7. Source checkpoint

The model this notebook adapts. 130 MB for ResNet-50, 492 MB for MaxViT.

In [ ]:
from huggingface_hub import hf_hub_download
import shutil, pathlib

ckpt = hf_hub_download(repo_id=arm["hf"], filename="seed0/best.pt")
dest = pathlib.Path(arm["run"])
dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(ckpt, dest)
print(f"{dest} ({dest.stat().st_size / 1e6:.0f} MB)")

## 8. Check the split

This reads the committed split rather than drawing a new one. Rebuilding it would
give a different test set, and every number already reported against it — including
the zero-shot run — would stop being comparable.

In [ ]:
import json

split = json.load(open("data/labels/rq2_splits.json"))
wf = split["water_fraction"]
mean = lambda ids: sum(wf[c] for c in ids) / len(ids)

print(f"test {split['test_size']} chips (mean water {mean(split['test']):.3f})")
print(f"pool {split['pool_size']} chips (mean water {mean(split['pool']):.3f})")
for seed, subsets in split["subsets"].items():
    print("  seed", seed, " ".join(f"n={n}:{mean(ids):.3f}" for n, ids in subsets.items()))

# Stratification is the point: water fraction stays flat across every subset,
# against a per-chip range of 0.00 to 0.90.

## 9. Run the sweep

13 runs: one un-adapted evaluation plus 4 sizes x 3 seeds.

`--source-config` and `--source-checkpoint` are what point the sweep at the arm you
chose in cell 5; without them it would fine-tune whatever `configs/finetune.yaml`
names, which is the ResNet-50 arm.

Batch 8 fits a T4 for ResNet-50, 4 for MaxViT. Raising `--workers` matters more than
batch size here, because the chips are small and the GPU spends its time waiting on
GeoTIFF reads.

**If the session drops**, rerun this cell: finished runs are reused and the sweep
picks up where it stopped. `--force` redoes everything, and changing a setting that
would alter a result invalidates the affected runs on its own.

To rehearse the wiring before committing to the full run, add
`--sizes 0 5 --seeds 0 --steps 10` and check it finishes in under a minute.

In [ ]:
# Shell cells below substitute $NAME and friends from these Python variables.
BATCH      = 8 if ARM == "resnet50" else 4   # MaxViT at batch 8 does not fit a T4
NAME       = f"rq2_finetune_{ARM}"
SRC_CONFIG = arm["config"]
SRC_CKPT   = arm["run"]

!PYTHONPATH=src python -m nisar_flood.experiments.finetune_curve \
    --config configs/finetune.yaml \
    --name $NAME \
    --source-config $SRC_CONFIG \
    --source-checkpoint $SRC_CKPT \
    --batch-size $BATCH --workers 2 \
    --out $OUT

The sweep writes `curve.json`, `curve.png` and one `run_n{n}_seed{s}.json` per run,
each carrying its per-chip IoUs for the error analysis in #25.

## 10. Results

In [ ]:
import json

curve = json.load(open(f"{OUT}/curve.json"))
summary, metric = curve["summary"], curve["metric"]
baseline = summary["0"]["mean"]

print(f"{metric}, {curve['steps']} steps, {len(curve['test_chips'])} test chips\n")
print(f"{'n':>4}  {'mean':>6}  {'std':>6}  {'vs n=0':>7}")
for n in sorted(summary, key=int):
    s = summary[n]
    print(f"{n:>4}  {s['mean']:.4f}  {s['std']:.4f}  {s['mean'] - baseline:+.4f}")

In [ ]:
from IPython.display import Image, display

display(Image(f"{OUT}/curve.png"))

### Reading it honestly

- At **n=35 the pool is exhausted**, so all three seeds train on the same chips. The
  spread there is training randomness alone, not the draw, and its error bar is not
  comparable to the ones at n=5 and n=10.
- The **n=0 point is this curve's anchor, not the RQ1 result**. The reportable
  zero-shot number is Nidhi's #22, which also splits open water from flooded
  vegetation and compares against a dB threshold. Quote hers, not this one.
- **"N chips recover X% of the gap"** needs the gap, which is the source model's
  Sentinel-1 score minus its zero-shot NISAR score. Fill that in once #22 lands.

## 11. Keep the results

Colab deletes everything when the session ends. The JSON and PNG are small, so pull
them out before you close the tab.

In [ ]:
import shutil

archive = shutil.make_archive(f"rq2_{ARM}", "zip", OUT)
print(archive, f"{pathlib.Path(archive).stat().st_size / 1e6:.1f} MB")

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not on Colab - the archive is in the working directory.")

To put the numbers into the repo, unzip the archive into
`reports/rq2_finetune_<arm>/` on the `feat/24-rq2-splits` branch and commit
`curve.json`, `curve.png` and the per-run files. The `runs/` directory is
gitignored, which is why results are copied into `reports/` rather than committed
where they were written.